# 05. CNN으로 MNIST 분류

손글씨 숫자(28×28 흑백) 이미지를 합성곱 신경망(CNN)으로 분류합니다.

**학습 목표**
- `torchvision.datasets`로 표준 데이터셋을 내려받고 정규화한다
- `Conv2d`, `MaxPool2d`, `Dropout`으로 CNN을 구성한다
- 학습/평가 함수를 분리하고, `state_dict`로 모델을 저장하고 불러온다

데이터: `data/downloads/` (MNIST, 첫 실행 시 자동 다운로드, git 제외), 모델: `data/mnist_cnn.pt` (git 제외)

In [ ]:
import torch

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__}, device: {device}")

## 1. 데이터 불러오기

`Normalize`의 평균과 표준편차(0.1307, 0.3081)는 MNIST 학습 데이터에서 구한 값입니다.

In [ ]:
from pathlib import Path
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

DATA = Path("data")
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])
train_ds = datasets.MNIST(DATA / "downloads", train=True, download=True, transform=transform)
test_ds = datasets.MNIST(DATA / "downloads", train=False, download=True, transform=transform)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=1000)
print(f"train {len(train_ds)} | test {len(test_ds)} | 이미지 shape {train_ds[0][0].shape}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 8, figsize=(10, 3))
for ax, i in zip(axes.flat, range(16)):
    img, label = train_ds[i]
    ax.imshow(img.squeeze(), cmap="gray")
    ax.set_title(label)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 2. CNN 모델

| 레이어 | 출력 shape |
|---|---|
| 입력 | 1×28×28 |
| Conv(32, 3×3) + ReLU + MaxPool | 32×14×14 |
| Conv(64, 3×3) + ReLU + MaxPool | 64×7×7 |
| Flatten → Linear(128) + Dropout | 128 |
| Linear(10) | 10 |

In [ ]:
from torch import nn

class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128), nn.ReLU(), nn.Dropout(0.25),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

model = CNN().to(device)
print(model)
print("파라미터 수:", sum(p.numel() for p in model.parameters()))

## 3. 학습과 평가

In [ ]:
from torch import optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

def train_one_epoch():
    model.train()
    total_loss = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y)
    return total_loss / len(train_ds)

def evaluate():
    model.eval()
    correct = 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            correct += (model(x).argmax(1) == y).sum().item()
    return correct / len(test_ds)

EPOCHS = 2   # CPU에서도 몇 분 안에 끝나도록 짧게
for epoch in range(1, EPOCHS + 1):
    loss = train_one_epoch()
    print(f"epoch {epoch} | train loss {loss:.4f} | test acc {evaluate():.4f}")

## 4. 예측 확인

틀린 예측은 빨간색으로 표시합니다.

In [ ]:
x, y = next(iter(test_loader))
model.eval()
with torch.no_grad():
    pred = model(x.to(device)).argmax(1).cpu()

fig, axes = plt.subplots(2, 8, figsize=(10, 3))
for ax, i in zip(axes.flat, range(16)):
    ax.imshow(x[i].squeeze(), cmap="gray")
    ax.set_title(f"{pred[i].item()} ({y[i].item()})", color="green" if pred[i] == y[i] else "red")
    ax.axis("off")
plt.suptitle("prediction (label)")
plt.tight_layout()
plt.show()

In [ ]:
wrong = (pred != y).nonzero().flatten()
print(f"이 배치에서 틀린 개수: {len(wrong)} / {len(y)}")

## 5. 모델 저장과 불러오기

모델 객체 전체가 아니라 파라미터(`state_dict`)만 저장하는 것이 권장 방식입니다.

In [ ]:
model_path = DATA / "mnist_cnn.pt"
torch.save(model.state_dict(), model_path)
print("저장:", model_path, f"({model_path.stat().st_size / 1024:.0f} KB)")

loaded = CNN().to(device)
loaded.load_state_dict(torch.load(model_path, map_location=device))
loaded.eval()
with torch.no_grad():
    same = torch.equal(loaded(x.to(device)).argmax(1).cpu(), pred)
print("불러온 모델의 예측이 같은가:", same)

## 요약
- `torchvision.datasets`와 `transforms`로 데이터를 내려받고 정규화한다
- CNN은 합성곱과 풀링으로 특징을 뽑고, 완전연결층으로 분류한다
- 2 epoch만으로도 테스트 정확도 약 98~99%를 얻는다
- `torch.save(model.state_dict())` / `load_state_dict()`로 모델을 저장하고 복원한다